In [1]:
## Data utilizada: # https://drive.google.com/drive/u/1/folders/1EoxsYIJd2mc5oRf2ZlbAS--bhw5E2MXa  Repertorio A
## Busca en la ambientes el que se llama ig-pipeline
## La primera vez debes reproducir el ambiente en la consola con uv sync (luego de esto ya va a estar disponible siempre)
## Luego solo lo buscas en Python enviroments (aunque se deberia seleccionar solo!)

import matplotlib.pyplot as plt
from seaborn import axes_style
import polars.selectors as cs
import seaborn.objects as so
import seaborn as sns
import polars as pl
import glob



pl.Config.set_tbl_rows(15)

polars.config.Config

In [ ]:
# La siguiente función categoriza el espacio clonal en diferentes rangos según el porcentaje de clones presentes en la muestra.

def categorize_clone(clonal_space):
	if(clonal_space<= 0.01):
		return "Small (<=0.01%)"
	elif(clonal_space > 0.01 and clonal_space <= 0.1):
		return "Medium (0.01%-0.1%)"
	elif(clonal_space > 0.1 and clonal_space <= 0.5):
		return "Large (0.1%-0.5%)"
	elif(clonal_space > 0.5 and clonal_space <= 1):
		return "Expanded (0.5%-1%)"
	elif(clonal_space > 1):
		return "Hyperexpanded (>1%)"

# Orden para graficar las categorías
category_order = {
    "Small (<=0.01%)": 1,
    "Medium (0.01%-0.1%)": 2,
    "Large (0.1%-0.5%)": 3,
    "Expanded (0.5%-1%)": 4,
    "Hyperexpanded (>1%)": 5
}

# Colores por categoría 
colormap = {
        "Small (<=0.01%)": "#8bafd9",
        "Medium (0.01%-0.1%)": "#6a88ab",
        "Large (0.1%-0.5%)": "#4a6482",
        "Expanded (0.5%-1%)": "#c45d88",
        "Hyperexpanded (>1%)": "#9e647c",
    }

# Orden en el que se van a mostrar en el eje X
order_samples =[
	"A_100",
	"A_200",
 	"A_400",
 	"A_800",
 	"A_1600",
 	"A_3200",
 	"A_6400",
 	"A_12800",
 	"A_25600",
 	"A_51200",
 	"A_102400"


]

# Modificar en caso de ser necesario
## Cantidad minima de lecturas para considerar un clon como válido
## !!! OJO QUE LOS RESULTADOS CON TUS DATOS CAMBIAN BASTANTE SI ES QUE ES MIN 1 O MIN 2
min_nreads_per_clone = 1
## Flag para indicar si se analizaran todas las lecturas o solo las productivas. 
## En caso de ser True, se analizaran solo las productivas. En caso de ser False, se analizaran todas las lecturas.
filter_by_productive = True

In [ ]:
"""
Función para graficar el espacio clonal categorizado por muestra o grupo.
@param df_cat: DataFrame de polars con los datos categorizados. Columnas: 
  - sample, 
  - clon_type: categorias definidas en categorize_clone
  - clonal_space_occupied: porcentaje de ocupancia
  - order: orden de aparición de las barras
  - n_reads_max_cat: cantidad máxima de lecturas por categoría 
@param save: booleano que indica si se desea guardar la figura. Por defecto es False.
@param name: nombre para guardar la figura. Por defecto es "plot".
"""
def plot_categorized_clones(df_cat,show_by,save=False,name="plot"):
    plt.rcParams["figure.dpi"] = 300

    #df_category_summary_gby = df_cat.sort(show_by)
    order_dict = {nombre: i for i, nombre in enumerate(order_samples, start=1)}    
    df_category_summary_gby = df_cat.with_columns(
        order_ = pl.col("sample").replace(order_dict).cast(pl.Int64)
    )

    plot = (
        so.Plot(
            df_category_summary_gby.sort(["order_","order"]),
            x="sample",
            y="clonal_space_occupied",
            color="clon_type",
            #text="n_reads_max_cat"
        )
        .add(so.Bar(alpha=0.8, width=0.8), so.Stack(), legend=True)
        .scale(color=colormap)
        .layout(size=(12, 4), engine="tight")
    ).theme(
        {
            "axes.edgecolor": "#95999d",
            "axes.spines.right": False,
            "axes.spines.top": False,
        }
    )

    #plot = plot.facet(show_by).share(x=False).layout(engine="constrained")
    plot = plot.share(x=False).layout(engine="constrained")

    plot.config.theme.update(axes_style("white"))
    fig = plot.plot()
    if(save==True):
        plot.save(f"{basedir}/figures/{name}.svg", dpi=300, format='svg', bbox_inches='tight')
    return plot

In [ ]:
## Input del notebook:
###- Múltiples archivos de clones: uno por cada muestra 
###    - (columnas: sequence_id, sequence, rev_comp, productive, ...., clone_id, etc) 
### Ejemplo: repertorio_A_insilico_30_seqs_clone-pass

basedir="../data/repertorio_A_clones/"
